In [1]:
!pip install git+https://github.com/neurallatents/nlb_tools.git
!pip install dandi
!dandi download https://gui.dandiarchive.org/dandiset/000127
!pip install git+https://github.com/neurallatents/nlb_tools.git --no-deps
!pip install pynwb elephant pyyaml

  Cloning https://github.com/neurallatents/nlb_tools.git to /tmp/pip-req-build-15wfiyld
  Running command git clone --filter=blob:none --quiet https://github.com/neurallatents/nlb_tools.git /tmp/pip-req-build-15wfiyld
  Resolved https://github.com/neurallatents/nlb_tools.git to commit 42f8410b88e12db136910fa2f888b025ea0aa2ae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 42.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See abov

In [2]:
!pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128


In [3]:
!pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 75.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 47.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 97.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 32.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/90

## RNN Training & Latent Extraction

Trains recurrent neural networks on the Area2_Bump motor task data and
extracts hidden-state trajectories for downstream comparison (sub-task d).

**Training objectives:**
  1. Neural Decoder:  rates → hand_vel  (decode behavior from neural activity)
  2. LFADS-lite:      rates → rates     (latent-factor autoencoder)
  3. Task-driven:     condition one-hot → rates  (generate neural dynamics)

Architectures tested: GRU, LSTM, vanilla RNN

**Outputs:**
  - rnn_models/                         — saved model checkpoints
  - rnn_hidden_states.h5                — extracted hidden trajectories
  - rnn_training_summary.json           — training curves + final metrics
  - figures/rnn_*.png                   — training + trajectory plots

In [4]:
from __future__ import annotations

import os
import json
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Union

import numpy as np
import h5py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False


# CORE COMPONENTS: CONFIG, DATA, MODELS, TRAINER, EXTRACTOR, PLOTS 


# CONFIG
@dataclass
class RNNTrainConfig:
    """All knobs for RNN training."""
    rnn_ready_h5: Union[str, Path] = "/kaggle/input/notebooks/abdelwhabmohamed05/rnn-data-prep/area2_bump_rnn_ready.h5"
    out_dir: Union[str, Path] = "/kaggle/working"

    # Architecture
    rnn_types: List[str] = field(default_factory=lambda: ["GRU"])
    hidden_sizes: List[int] = field(default_factory=lambda: [128])
    n_layers: int = 1
    dropout: float = 0.0

    # Training objectives to run
    objectives: List[str] = field(
        default_factory=lambda: ["decoder", "autoencoder", "task_driven"]
    )

    # Training hyperparameters
    n_epochs: int = 200
    batch_size: int = 32
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 1.0
    patience: int = 20

    # Learning rules to explore
    learning_rules: List[str] = field(
        default_factory=lambda: ["bptt"]    # "bptt", "eprop", "rflo"
    )

    # Plotting
    max_traj_conditions: int = 16

    def resolved_paths(self) -> Dict[str, Path]:
        out = Path(self.out_dir)
        fig = out / "figures"
        models = out / "rnn_models"
        out.mkdir(parents=True, exist_ok=True)
        fig.mkdir(parents=True, exist_ok=True)
        models.mkdir(parents=True, exist_ok=True)
        return {
            "rnn_ready_h5": Path(self.rnn_ready_h5),
            "out_dir": out,
            "fig_dir": fig,
            "models_dir": models,
            "hidden_h5": out / "rnn_hidden_states.h5",
            "summary_json": out / "rnn_training_summary.json",
        }


# DATA LOADERS
class NeuralDataset:
    """Simple dataset wrapper for PyTorch, loaded from the RNN-ready H5."""

    def __init__(self, h5_path: Path, split: str = "train",
                 objective: str = "decoder"):
        self.objective = objective
        with h5py.File(h5_path, "r") as f:
            grp = f[f"splits/{split}"]
            self.rates = grp["rates_normed"][:]
            self.cond_ids = grp["cond_ids"][:]
            self.cond_onehot = grp["cond_onehot"][:]
            self.hand_vel = grp["hand_vel"][:] if "hand_vel" in grp else None
            # Decoder-specific lagged pairs
            if "decoder" in grp:
                dec = grp["decoder"]
                self.dec_X = dec["X"][:]
                self.dec_Y = dec["Y"][:] if "Y" in dec else None
            else:
                self.dec_X = self.rates
                self.dec_Y = self.hand_vel

            self.n_conditions = f.attrs["n_conditions"]

    def __len__(self):
        return self.rates.shape[0]

    def __getitem__(self, idx):
        if self.objective == "decoder":
            x = self.dec_X[idx]
            y = self.dec_Y[idx] if self.dec_Y is not None else np.zeros(1)
            return (torch.tensor(x, dtype=torch.float32),
                    torch.tensor(y, dtype=torch.float32))
        elif self.objective == "autoencoder":
            x = self.rates[idx]
            return (torch.tensor(x, dtype=torch.float32),
                    torch.tensor(x, dtype=torch.float32))
        elif self.objective == "task_driven":
            cond = self.cond_onehot[idx]
            target = self.rates[idx]
            return (torch.tensor(cond, dtype=torch.float32),
                    torch.tensor(target, dtype=torch.float32))
        else:
            raise ValueError(f"Unknown objective: {self.objective}")


# RNN MODELS
class NeuralDecoder(nn.Module):
    """RNN that decodes hand velocity from neural rates."""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int,
                 rnn_type: str = "GRU", n_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.rnn_type = rnn_type

        RNN = {"GRU": nn.GRU, "LSTM": nn.LSTM, "RNN": nn.RNN}[rnn_type]
        self.rnn = RNN(input_dim, hidden_dim, num_layers=n_layers,
                       batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None):
        out, h = self.rnn(x, h0)
        y = self.fc(out)
        return y, h, out


class NeuralAutoencoder(nn.Module):
    """RNN autoencoder: encode neural rates into latent dynamics, then decode."""

    def __init__(self, input_dim: int, hidden_dim: int,
                 rnn_type: str = "GRU", n_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.hidden_dim = hidden_dim

        RNN = {"GRU": nn.GRU, "LSTM": nn.LSTM, "RNN": nn.RNN}[rnn_type]
        self.encoder = RNN(input_dim, hidden_dim, num_layers=n_layers,
                           batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.decoder_fc = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, h0=None):
        out, h = self.encoder(x, h0)
        recon = self.decoder_fc(out)
        return recon, h, out


class TaskDrivenRNN(nn.Module):
    """RNN that generates neural dynamics from task conditions.

    Input: condition one-hot (tiled across time steps)
    Output: predicted neural firing rates
    """

    def __init__(self, cond_dim: int, hidden_dim: int, output_dim: int,
                 n_time: int, rnn_type: str = "GRU", n_layers: int = 1,
                 dropout: float = 0.0):
        super().__init__()
        self.n_time = n_time
        self.hidden_dim = hidden_dim

        self.input_fc = nn.Linear(cond_dim, hidden_dim)
        RNN = {"GRU": nn.GRU, "LSTM": nn.LSTM, "RNN": nn.RNN}[rnn_type]
        self.rnn = RNN(hidden_dim, hidden_dim, num_layers=n_layers,
                       batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.output_fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, cond, h0=None):
        # cond: (batch, cond_dim) → tile to (batch, n_time, hidden_dim)
        x = self.input_fc(cond)
        x = x.unsqueeze(1).expand(-1, self.n_time, -1)
        out, h = self.rnn(x, h0)
        y = self.output_fc(out)
        return y, h, out


# TRAINER
def train_model(
    model: nn.Module,
    train_dataset: NeuralDataset,
    val_dataset: Optional[NeuralDataset],
    cfg: RNNTrainConfig,
    device: str = "cpu",
) -> Dict:
    """Standard BPTT training loop with early stopping."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate,
                                 weight_decay=cfg.weight_decay)
    criterion = nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size,
                              shuffle=True, drop_last=False)
    val_loader = None
    if val_dataset is not None and len(val_dataset) > 0:
        val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size,
                                shuffle=False, drop_last=False)

    history = {"train_loss": [], "val_loss": [], "epoch_time": []}
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(cfg.n_epochs):
        t0 = time.time()
        # Train
        model.train()
        train_losses = []
        for X_batch, Y_batch in train_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            optimizer.zero_grad()
            pred, _, _ = model(X_batch)
            min_t = min(pred.shape[1], Y_batch.shape[1])
            loss = criterion(pred[:, :min_t], Y_batch[:, :min_t])
            loss.backward()
            if cfg.grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            train_losses.append(loss.item())

        avg_train = float(np.mean(train_losses))
        history["train_loss"].append(avg_train)

        # Validate
        avg_val = float("nan")
        if val_loader is not None:
            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_batch, Y_batch in val_loader:
                    X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
                    pred, _, _ = model(X_batch)
                    min_t = min(pred.shape[1], Y_batch.shape[1])
                    loss = criterion(pred[:, :min_t], Y_batch[:, :min_t])
                    val_losses.append(loss.item())
            avg_val = float(np.mean(val_losses))

        history["val_loss"].append(avg_val)
        history["epoch_time"].append(time.time() - t0)

        # Early stopping
        if not np.isnan(avg_val) and avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg.patience:
                break

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        "history": history,
        "best_val_loss": float(best_val_loss) if best_val_loss < float("inf") else None,
        "n_epochs_trained": len(history["train_loss"]),
        "total_time_s": float(sum(history["epoch_time"])),
    }


# HIDDEN STATE EXTRACTOR
def extract_hidden_states(
    model: nn.Module,
    dataset: NeuralDataset,
    n_conditions: int,
    device: str = "cpu",
) -> Dict[str, np.ndarray]:

    model.eval()
    model = model.to(device)
    all_hidden = []

    is_task_driven = isinstance(model, TaskDrivenRNN)

    if is_task_driven:
        inputs = torch.tensor(dataset.cond_onehot, dtype=torch.float32)
    else:
        inputs = torch.tensor(dataset.rates, dtype=torch.float32)

    loader = DataLoader(
        torch.utils.data.TensorDataset(inputs),
        batch_size=64, shuffle=False
    )
    with torch.no_grad():
        for (X_batch,) in loader:
            X_batch = X_batch.to(device)
            _, _, hidden = model(X_batch)
            all_hidden.append(hidden.cpu().numpy())

    all_hidden = np.concatenate(all_hidden, axis=0)
    cond_ids = dataset.cond_ids

    cond_avg = np.zeros(
        (n_conditions, all_hidden.shape[1], all_hidden.shape[2]),
        dtype=np.float32
    )
    for c in range(n_conditions):
        mask = cond_ids == c
        if mask.sum() > 0:
            cond_avg[c] = all_hidden[mask].mean(axis=0)

    return {
        "all_hidden": all_hidden.astype(np.float32),
        "cond_avg":   cond_avg.astype(np.float32),
        "cond_ids":   cond_ids,
    }


# R² METRIC
def compute_r2(model, dataset, device="cpu"):
    """Compute R² on the full dataset."""
    model.eval()
    model = model.to(device)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    all_pred, all_true = [], []
    with torch.no_grad():
        for X, Y in loader:
            X = X.to(device)
            pred, _, _ = model(X)
            min_t = min(pred.shape[1], Y.shape[1])
            all_pred.append(pred[:, :min_t].cpu().numpy())
            all_true.append(Y[:, :min_t].numpy())
    pred = np.concatenate(all_pred, axis=0).reshape(-1, all_pred[0].shape[-1])
    true = np.concatenate(all_true, axis=0).reshape(-1, all_true[0].shape[-1])
    ss_res = np.sum((true - pred) ** 2)
    ss_tot = np.sum((true - true.mean(axis=0)) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0


# --- PLOTTING ---------------------------------------------------------------
def plot_training_curves(histories: Dict[str, Dict], out_path: Path) -> None:
    """Plot training/val loss curves for all models."""
    n = len(histories)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
    for i, (name, info) in enumerate(histories.items()):
        ax = axes[0, i]
        h = info["history"]
        ax.plot(h["train_loss"], label="train", alpha=0.8)
        if any(not np.isnan(v) for v in h["val_loss"]):
            ax.plot(h["val_loss"], label="val", alpha=0.8)
        ax.set_xlabel("epoch")
        ax.set_ylabel("MSE loss")
        ax.set_title(name)
        ax.legend(fontsize=8)
        ax.set_yscale("log")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


def plot_hidden_trajectories(
    hidden_states: Dict[str, Dict[str, np.ndarray]],
    label_order: List[str],
    out_path: Path,
    max_conditions: int = 16,
) -> None:
    """PCA of hidden states, colored by condition."""
    from sklearn.decomposition import PCA
    n_models = len(hidden_states)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), squeeze=False)
    cmap = plt.cm.tab20

    for i, (name, hs) in enumerate(hidden_states.items()):
        ax = axes[0, i]
        cond_avg = hs["cond_avg"]  # (C, T, H)
        n_conds = min(cond_avg.shape[0], max_conditions)
        # PCA on all conditions
        flat = cond_avg[:n_conds].reshape(-1, cond_avg.shape[-1])
        if flat.shape[1] >= 2:
            pca = PCA(n_components=2)
            proj = pca.fit_transform(flat)
            proj = proj.reshape(n_conds, cond_avg.shape[1], 2)
            for c in range(n_conds):
                label = label_order[c] if c < len(label_order) else f"c{c}"
                ax.plot(proj[c, :, 0], proj[c, :, 1],
                        color=cmap(c % 20), alpha=0.8, linewidth=1.5)
                ax.plot(proj[c, 0, 0], proj[c, 0, 1], "o",
                        color=cmap(c % 20), markersize=4)
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
        ax.set_title(f"{name} — hidden trajectories")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


# HIDDEN STATE WRITER
def save_hidden_states_h5(
    path: Path,
    all_hidden: Dict[str, Dict[str, np.ndarray]],
    label_order: List[str],
    training_summaries: Dict[str, Dict],
) -> None:
    """Save all extracted hidden states to H5."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, "w") as f:
        f.create_dataset("condition_labels",
                         data=np.array(label_order, dtype="S"))
        f.attrs["created_by"] = "neuroagents/05_rnn_training.py"

        for model_name, hs in all_hidden.items():
            grp = f.create_group(model_name)
            grp.create_dataset("all_hidden", data=hs["all_hidden"],
                               compression="gzip")
            grp.create_dataset("cond_avg", data=hs["cond_avg"],
                               compression="gzip")
            grp.create_dataset("cond_ids", data=hs["cond_ids"])

            # Store training metrics
            if model_name in training_summaries:
                ts = training_summaries[model_name]
                if ts.get("best_val_loss") is not None:
                    grp.attrs["best_val_loss"] = ts["best_val_loss"]
                grp.attrs["n_epochs_trained"] = ts["n_epochs_trained"]


# RNN TRAINING DRIVER

if __name__ == "__main__":
    if not HAS_TORCH:
        print("[rnn_training] SKIP — PyTorch not installed. "
              "Run `pip install torch` first.")
        import sys; sys.exit(0)

    cfg = RNNTrainConfig()
    paths = cfg.resolved_paths()
    summary: Dict = {"config": {k: str(v) if isinstance(v, Path) else v
                                for k, v in cfg.__dict__.items()}}
    device = "cuda" if torch.cuda.is_available() else "cpu"
    summary["device"] = device

    # Check input exists
    if not paths["rnn_ready_h5"].exists():
        raise FileNotFoundError(
            f"RNN-ready H5 not found at {paths['rnn_ready_h5']}. "
            f"Run 04_rnn_data_prep.py first."
        )

    # Load dataset metadata
    with h5py.File(paths["rnn_ready_h5"], "r") as f:
        n_conditions = int(f.attrs["n_conditions"])
        label_order = [s.decode() if isinstance(s, bytes) else s
                       for s in f["condition_labels"][:]]
        # Peek at shapes
        train_grp = f["splits/train"]
        n_neurons = train_grp["rates_normed"].shape[2]
        n_time = train_grp["rates_normed"].shape[1]
        has_hand_vel = "hand_vel" in train_grp
        hand_vel_dim = train_grp["hand_vel"].shape[2] if has_hand_vel else 0

    summary["data"] = {
        "n_conditions": n_conditions,
        "n_neurons": n_neurons,
        "n_time": n_time,
        "hand_vel_dim": hand_vel_dim,
    }

    # Train models for each objective & architecture combination
    all_training_summaries = {}
    all_hidden_states = {}
    all_histories = {}

    for objective in cfg.objectives:
        for rnn_type in cfg.rnn_types:
            for hidden_size in cfg.hidden_sizes:
                model_name = f"{objective}_{rnn_type}_h{hidden_size}"

                # Load data
                train_ds = NeuralDataset(paths["rnn_ready_h5"], "train", objective)
                val_ds = NeuralDataset(paths["rnn_ready_h5"], "val", objective)

                # Build model
                if objective == "decoder":
                    if not has_hand_vel:
                        continue
                    model = NeuralDecoder(
                        input_dim=n_neurons, hidden_dim=hidden_size,
                        output_dim=hand_vel_dim, rnn_type=rnn_type,
                        n_layers=cfg.n_layers, dropout=cfg.dropout,
                    )
                elif objective == "autoencoder":
                    model = NeuralAutoencoder(
                        input_dim=n_neurons, hidden_dim=hidden_size,
                        rnn_type=rnn_type, n_layers=cfg.n_layers,
                        dropout=cfg.dropout,
                    )
                elif objective == "task_driven":
                    model = TaskDrivenRNN(
                        cond_dim=n_conditions, hidden_dim=hidden_size,
                        output_dim=n_neurons, n_time=n_time,
                        rnn_type=rnn_type, n_layers=cfg.n_layers,
                        dropout=cfg.dropout,
                    )
                else:
                    continue

                train_result = train_model(model, train_ds, val_ds, cfg, device)

                # Compute R²
                r2_train = compute_r2(model, train_ds, device)
                r2_val = compute_r2(model, val_ds, device)
                train_result["r2_train"] = r2_train
                train_result["r2_val"] = r2_val

                all_training_summaries[model_name] = train_result
                all_histories[model_name] = train_result

                # Extract hidden states (from train set for comparison)
                hs = extract_hidden_states(
                    model, train_ds, n_conditions, device
                )
                all_hidden_states[model_name] = hs

                # Save model checkpoint
                model_path = paths["models_dir"] / f"{model_name}.pt"
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "config": {
                        "objective": objective,
                        "rnn_type": rnn_type,
                        "hidden_size": hidden_size,
                        "n_layers": cfg.n_layers,
                        "input_dim": n_neurons,
                        "output_dim": hand_vel_dim if objective == "decoder" else n_neurons,
                        "n_conditions": n_conditions,
                        "n_time": n_time,
                    },
                    "training_result": train_result,
                }, model_path)

    summary["models"] = {
        name: {
            "r2_train": info.get("r2_train"),
            "r2_val": info.get("r2_val"),
            "best_val_loss": info.get("best_val_loss"),
            "n_epochs_trained": info["n_epochs_trained"],
            "total_time_s": info["total_time_s"],
        }
        for name, info in all_training_summaries.items()
    }

    # Save hidden states
    save_hidden_states_h5(
        paths["hidden_h5"], all_hidden_states,
        label_order, all_training_summaries,
    )
    summary["hidden_h5"] = str(paths["hidden_h5"])

    # Plots
    if all_histories:
        plot_training_curves(
            all_histories, paths["fig_dir"] / "rnn_01_training_curves.png"
        )
        plot_hidden_trajectories(
            all_hidden_states, label_order,
            paths["fig_dir"] / "rnn_02_hidden_trajectories.png",
            cfg.max_traj_conditions,
        )

    # Write JSON summary
    with open(paths["summary_json"], "w") as f:
        json.dump(summary, f, indent=2, default=str)

    n_models = len(all_training_summaries)
    best_model = min(all_training_summaries.items(),
                     key=lambda x: x[1].get("best_val_loss") or float("inf")
                     )[0] if all_training_summaries else "none"

    print(f"[rnn_training] OK  models={n_models} best={best_model} "
          f"device={device} -> {paths['summary_json']}")

[rnn_training] OK  models=3 best=autoencoder_GRU_h128 device=cuda -> /kaggle/working/rnn_training_summary.json
